# Ladin Bridge-Transfer — Colab Orchestrator

Runs the full experiment matrix on a T4 GPU.
All results and adapters are checkpointed to Google Drive after each stage.

In [ ]:
# ── 1. Mount Google Drive ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

DRIVE_ROOT = '/content/drive/MyDrive/ladin-bridge-transfer'
import os; os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive mounted. Root:', DRIVE_ROOT)

In [ ]:
# ── 2. Clone / pull the repo ─────────────────────────────────────────────────
import os

REPO_URL = "https://github.com/YOUR_USERNAME/ladin-bridge-transfer"  # ← replace YOUR_USERNAME
REPO_DIR = "/content/ladin-bridge-transfer"

if os.path.isdir(REPO_DIR):
    # Already cloned (e.g. after a Colab disconnect) — just pull latest
    %cd $REPO_DIR
    !git pull
else:
    !git clone $REPO_URL $REPO_DIR
    %cd $REPO_DIR

!ls -la

In [ ]:
# ── 3. Install dependencies ──────────────────────────────────────────────────
# Checks if packages are already present AND at required versions; if not,
# installs/upgrades them and restarts the runtime so changes load cleanly.
def _deps_ok():
    try:
        import transformers, peft, datasets, accelerate, sacrebleu
        import torchao
        from packaging.version import Version
        if Version(torchao.__version__) < Version("0.16.0"):
            print(f"torchao {torchao.__version__} is too old (need >=0.16.0) — upgrading…")
            return False
        return True
    except ImportError:
        return False

if _deps_ok():
    print("Dependencies already installed — skipping.")
else:
    import subprocess, sys
    r = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        capture_output=True, text=True,
    )
    if r.returncode != 0:
        print(r.stderr[-3000:])
        raise RuntimeError("pip install failed — see errors above")
    print("Dependencies installed. Restarting runtime to load them cleanly…")
    print("▶ After the restart, re-run all cells from cell 1.")
    import IPython
    IPython.get_ipython().kernel.do_shutdown(restart=True)


In [ ]:
# ── 4. HuggingFace login (token from Colab Secrets) ─────────────────────────
from google.colab import userdata
import huggingface_hub

hf_token = userdata.get('HF_TOKEN')   # set in Colab Secrets (key icon in sidebar)
huggingface_hub.login(token=hf_token, add_to_git_credential=False)
print('HuggingFace login complete.')

In [ ]:
# ── 5. Verify GPU ────────────────────────────────────────────────────────────
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')

In [ ]:
# ── 6. Configure logging ──────────────────────────────────────────────────────
import logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(name)s | %(message)s',
    datefmt='%H:%M:%S',
)
# Cache HF datasets to Drive so re-download is avoided on reconnect
import os
HF_CACHE = f'{DRIVE_ROOT}/hf_cache'
os.makedirs(HF_CACHE, exist_ok=True)
os.environ['HF_DATASETS_CACHE'] = HF_CACHE
os.environ['TRANSFORMERS_CACHE'] = HF_CACHE
print('HF cache dir:', HF_CACHE)

In [ ]:
# ── 7. Run the full experiment matrix ────────────────────────────────────────
#
# SMOKE RUN: subset_size=500, epochs_ladin=1 (see configs/default.yaml)
# To run for real: set subset_size=18139, epochs_ladin=3 in configs/default.yaml

from src.experiments.run_all import run_experiment

run_experiment(
    drive_root=DRIVE_ROOT,
    run_zero_shot=True,
    run_direct_cond=True,
    run_bridges=True,
    run_similarity=True,
    run_ablation=False,  # SDLad-Ita repo not yet confirmed
    cache_dir=HF_CACHE,
)

In [ ]:
# ── 8. Quick sanity check: print main results table ──────────────────────────
import pandas as pd
from pathlib import Path

results_csv = Path(DRIVE_ROOT) / 'results' / 'main_results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    print(df.to_string(index=False))
else:
    print('main_results.csv not yet written — run step 7 first.')